# 01 · Understanding the data: a guided tour

This notebook is a **lesson**, not just an analysis. Phase 1 drew a set of conclusions that drive every later design choice. Here you will see **what each conclusion is, the evidence behind it, and the reasoning that connects them**.

### How every section is structured
| Marker | Meaning |
|---|---|
| 🎯 **Question** | What we wanted to know |
| 🤔 **Why it matters** | What would go wrong if we got this wrong |
| 🔬 **Method** | What the code does, in plain words |
| 📊 **What we saw** | The key numbers (the code cell prints them) |
| 💡 **Inference** | What we concluded, *why*, and which other explanations we ruled out |
| ➡️ **Consequence** | What we changed in the project (decision IDs refer to `docs/decisions.md`) |

### Sections
0. Poker vocabulary you need
1. The scoring rule: what does "good" mean?
2. What do the columns mean? Verifying the log
3. The world: tables, stakes, players
4. The labels: what we know and the traps
5. Exposure: does playing together a lot mean anything?
6. Evidence hands: what makes a hand suspicious?
7. Behavior toward the partner: the core signal
8. Walkthroughs: reading nine real evidence hands
9. Summary: the reasoning chain, open questions and a self-check

> Everything here uses competition data, so this notebook's **outputs stay local** (rules: "Competition Use Only"). The repo version is committed without outputs.

## 0 · Poker vocabulary you need

| Term | Meaning |
|---|---|
| **Six-max No-Limit Hold'em** | 6 players per hand. Each gets 2 private *hole cards*; up to 5 shared *board* cards; any bet size up to your whole stack. |
| **Button / SB / BB** | The button is the dealer seat. The two seats after it post forced bets: the **small blind** and **big blind**. |
| **bb (big blind)** | The unit of money in poker analysis. "Won 10 bb" means 10 × the big blind, so results compare across stakes. |
| **Streets** | *Preflop* (no board), *flop* (3 board cards), *turn* (4th), *river* (5th). |
| **Actions** | **fold** (give up), **check** (pass when no bet), **call** (match a bet), **bet** (first wager on a street), **raise** (increase a bet), **all-in** (put in every chip). |
| **Aggressor** | The player who made the latest bet or raise. They "set the price" others must respond to. |
| **Showdown** | Players still in at the end reveal cards; the best hand wins. |
| **Heads-up** | Only two players left in the hand. |
| **The three collusion families** | **Directed transfer** (chip dumping): one partner deliberately loses chips to the other. **Soft play**: partners avoid attacking each other. **Coordinated isolation**: partners squeeze third players out of pots with combined aggression. |
| **Squeeze / sandwich** | Player A bets, a third player calls or sits in between, and partner B re-raises. The third player is trapped between two aggressors and usually folds. |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from suspoker.config import RAW_DIR
from suspoker.handview import format_hand
from suspoker.metric import WEIGHTS, average_precision

pl.Config(tbl_rows=30, tbl_cols=20, tbl_width_chars=180, fmt_str_lengths=60)
FAMILIES = ["directed_transfer", "soft_play", "coordinated_isolation"]

hands = pl.read_parquet(RAW_DIR / "hands.parquet")
seats_lf = pl.scan_parquet(RAW_DIR / "seats.parquet")
actions_lf = pl.scan_parquet(RAW_DIR / "actions.parquet")
labels = pl.read_csv(RAW_DIR / "development_labels.csv")
evidence = pl.read_csv(RAW_DIR / "development_evidence.csv")
eval_pairs = pl.read_csv(RAW_DIR / "evaluation_pairs.csv")
player_table = (seats_lf.select("hand_id", "player_id").unique("player_id")
                .collect().join(hands.select("hand_id", "table_id"), on="hand_id").drop("hand_id"))

## 1 · The scoring rule: what does "good" mean?

🎯 **Question:** How exactly is a submission scored, and how much does each part count?

🤔 **Why it matters:** The score decides where effort pays off. The description listed three parts (Pair AP, Evidence MAP@5, Behavior MAP) but **never said how they combine**. Splitting effort evenly could waste most of it.

🔬 **Method:** We found the organizer's public reference-metric notebook on Kaggle, ported it line by line into `suspoker/metric.py`, and wrote a *parity test*. It runs our copy and the official code on 25 random submissions, including deliberate ties, and requires identical scores.

📊 **What we saw:** `final = 0.70 × Pair AP + 0.20 × Evidence MAP@5 + 0.10 × Behavior MAP`.

### A quick intuition for Average Precision (AP)
Sort all pairs by your risk score, highest first. Walk down the list. Every time you hit a true colluder, note your *precision so far* (colluders found ÷ pairs looked at). AP is the average of those notes. It rewards putting real colluders **at the top**. Run the cell below to see it.

In [ ]:
y = np.array([1, 0, 1, 0, 0])            # truth: pairs 1 and 3 are colluders
good = np.array([.9, .1, .8, .2, .3])    # both colluders ranked on top
meh  = np.array([.9, .8, .7, .2, .1])    # one innocent pair sneaks in at rank 2
bad  = np.array([.1, .9, .2, .8, .7])    # colluders ranked last
for name, s in [("good", good), ("meh", meh), ("bad", bad)]:
    print(f"{name:>5}: AP = {average_precision(y, s):.3f}")
print("weights:", WEIGHTS)
print("Ties: equal scores are ordered by pair_id, so a constant score is just an arbitrary ranking.")

💡 **Inference:**
- **Ranking pairs is 70% of the score.** A perfect evidence ranker on top of a mediocre pair ranker still scores poorly. Pair ranking gets the most modeling attention.
- AP only cares about **order**, not calibration. A risk score of 0.9 vs 0.6 doesn't matter, only which pairs come first. Calibration still matters for the website's "risk meter", though.
- **Evidence (20%) counts only for true colluders.** For innocent pairs, evidence hands cost nothing, so we always submit 5.
- **Behavior (10%) is "AP per family".** A pair's risk counts toward a family only if we *predicted* that family. If we never predict a family, its AP collapses. So every pair gets its most likely family, and the validator refuses a submission that never predicts one of them.

➡️ **Consequence:** DEC-007 (port the official metric, parity-tested; prioritize Pair AP). It also fixed SPEC §5.6, the behavior labeling policy.

## 2 · What do the columns mean? Verifying the log

🎯 **Question:** What exactly do `amount`, `amount_to`, `to_call`, `pot_before` and the rest mean?

🤔 **Why it matters:** Every feature ("folded while facing the partner's bet", "chips lost to the partner") is built from these columns. Misreading one silently corrupts every feature. And our own game must later **write data in exactly this format** so the same feature code works on it.

🔬 **Method: hypothesis → invariant → check.** For each guess, write down a rule that must hold *on every row* if the guess is right, and measure the share of rows where it holds. 100% means confirmed; anything less means the guess (or the check) is wrong. The full version (`scripts/check_semantics.py`) ran on 400,000 hands. Below is a quick version on 20,000.

In [ ]:
sub_ids = hands.head(20_000).select("hand_id")
s = seats_lf.join(sub_ids.lazy(), on="hand_id", how="semi").collect()
a = actions_lf.join(sub_ids.lazy(), on="hand_id", how="semi").collect().sort("hand_id", "action_no")
h = hands.head(20_000)

per_hand = s.group_by("hand_id").agg(pl.col("net_chips").sum().alias("net"), pl.col("total_contribution").sum().alias("contrib")).join(h, on="hand_id")
chain = a.with_columns(pl.col("pot_before").shift(-1).over("hand_id").alias("next")).filter(pl.col("next").is_not_null())
checks = {
    "money is conserved: sum(net_chips) = 0 per hand (no rake)": (per_hand["net"] == 0).mean(),
    "final_pot = sum of everyone's contribution": (per_hand["final_pot"] == per_hand["contrib"]).mean(),
    "pot grows by exactly each action's `amount`": (chain["next"] == chain["pot_before"] + chain["amount"]).mean(),
    "a call adds exactly `to_call` chips": a.filter(pl.col("action") == "call").select((pl.col("amount") == pl.col("to_call")).mean()).item(),
    "an all-in adds the whole remaining stack": a.filter(pl.col("action") == "all_in").select((pl.col("amount") == pl.col("stack_before")).mean()).item(),
    "folds and checks add 0 chips": a.filter(pl.col("action").is_in(["fold", "check"])).select((pl.col("amount") == 0).mean()).item(),
}
for k, v in checks.items():
    print(f"{v:8.2%}  {k}")
# one worked hand with an all-in
hid = a.filter(pl.col("action") == "all_in")["hand_id"][0]
print()
print(format_hand(hid, h, s, a))

📊 **What we saw (full 400k-hand run):** every invariant holds at **100%**. Along the way we learned:
- **`amount`** = chips *added by this action*. **`amount_to`** = the player's *total commitment on this street after the action*. In the worked hand, "raise to 216 (+216)" on the turn means that player's turn total became 216. Notice also that the two players at showdown both held K-J: it is a **split pot**, so each "wins" and their net is only +2 chips. A big pot with a tiny net result is a pattern our features must not mistake for a transfer.
- **Blinds are not action rows.** They are already inside `pot_before` (the first action's pot is SB + BB) and inside `total_contribution`.
- **`to_call`** = highest commitment on the street − your own, **capped at your stack**. You can't be asked for more than you have.
- **All-ins come in two kinds.** About 52% are *all-in calls* (`amount ≤ to_call`) and the rest are aggressive all-in bets or raises. Treating every all-in as aggression would be a mistake.
- **Uncalled bets stay in the pot.** If you bet and everyone folds, your bet still counts in `final_pot` and your contribution. The winner's `net_chips` nets it out, but **pot size overstates what was really contested**.
- Our own first check of `to_call` showed 95.9%, not 100%. The data was fine; *our formula* was wrong. Investigating the 4% exposed the bug. **A check below 100% is a question, not an answer.**

💡 **Inference:** The log is **complete and exact**. Combined with the fact that **every player's hole cards are visible**, we can reconstruct the full situation at every decision (price, pot odds, who is ahead) without re-running a poker engine. That makes features like "folded the best hand to the partner" directly measurable.

➡️ **Consequence:** DEC-009, `docs/data_semantics.md` (binding for our game's logger). Pot sizes are measured in bb and corrected for uncalled bets.

## 3 · The world: tables, stakes, players

🎯 **Question:** How is the population organized?

🤔 **Why it matters:** It decides which pairs can even exist, how to compare money across games, and how to split data for validation.

In [ ]:
print(hands.group_by("phase").len())
per_table = hands.group_by("table_id").agg(pl.len().alias("hands"), pl.col("big_blind").n_unique().alias("n_stakes"),
                                           pl.col("big_blind").first().alias("bb"))
print(per_table.select(pl.col("hands").min().alias("min_hands"), pl.col("hands").max().alias("max_hands"),
                       pl.col("n_stakes").max().alias("max_stakes_per_table")))
print(per_table.group_by("bb").len().sort("bb"))
print("players per table:", player_table.group_by("table_id").len()["len"].unique().to_list())
print("tables per player:", player_table.group_by("player_id").len()["len"].max())

In [ ]:
player_hands = seats_lf.group_by("player_id").len().collect()["len"]
stacks_bb = (seats_lf.select("hand_id", "starting_stack").collect()
             .join(hands.select("hand_id", "big_blind"), on="hand_id")
             .select((pl.col("starting_stack") / pl.col("big_blind")).alias("bb"))["bb"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(player_hands, bins=60); ax[0].set_title("hands played per player")
ax[1].hist(stacks_bb.clip(0, 400), bins=80); ax[1].set_title("starting stack in big blinds (clipped at 400)")
plt.show()
print(stacks_bb.describe())

📊 **What we saw:** 400 tables × exactly 5,000 hands. **Each table has one fixed stake** (bb = 2: 218 tables, 4: 143, 10: 39). **Exactly 30 players per table, and each player sits at only one table.** Players play 243–2,016 hands (median 987). Starting stacks run 40–250 bb (median ~108 bb).

💡 **Inference:**
1. **Pairs only exist inside a table.** Two players at different tables never meet, so the 400 tables are 400 separate small worlds of 435 possible pairs each (30 × 29 / 2).
2. **Money must be measured in bb.** A 100-chip pot is huge at bb = 2 and small at bb = 10. Raw chips would make high-stake tables look "suspicious" just because numbers are bigger. That would be a fake signal.
3. **Activity is uneven** (243 vs 2,016 hands). Some pairs get far more chances to interact, so raw counts are unfair across pairs (section 5 shows this matters).
4. Our game's 100 bb starting stack sits right in the typical range, so that choice is realistic.

➡️ **Consequence:** Features use bb. Candidate pairs are generated per table. The table becomes the natural unit for cross-validation (section 4).

## 4 · The labels: what we know and the traps

🎯 **Question:** What do the labels cover, and how could they mislead us?

🤔 **Why it matters:** Labels are what the model learns from *and* how we measure it. A mistake here gives a model that looks great locally and fails on Kaggle.

In [ ]:
print(labels.group_by("label_status", "behavior_family").len().sort("len", descending=True))
lab = (labels.join(player_table.rename({"player_id": "player_1", "table_id": "t1"}), on="player_1")
             .join(player_table.rename({"player_id": "player_2", "table_id": "t2"}), on="player_2"))
print("labeled pairs whose players sit at different tables:", (lab["t1"] != lab["t2"]).sum())
print("tables with >=1 labeled pair:", lab["t1"].n_unique(), "of", hands["table_id"].n_unique())
print("how many positive pairs a table has (count of tables):")
print(lab.filter(pl.col("label") == 1).group_by("t1").len()["len"].value_counts().sort("len"))

In [ ]:
long = pl.concat([labels.select("pair_id", "label", "behavior_family", pl.col("player_1").alias("player")),
                  labels.select("pair_id", "label", "behavior_family", pl.col("player_2").alias("player"))])
deg = long.group_by("player").agg(pl.len().alias("n_pairs"), pl.col("label").sum().alias("n_pos"))
print("labeled pairs per player  (n_pairs, players):", deg["n_pairs"].value_counts().sort("n_pairs").rows())
print("positive pairs per positive player (n_pos, players):", deg.filter(pl.col("n_pos") > 0)["n_pos"].value_counts().sort("n_pos").rows())
fam_per_player = long.filter(pl.col("label") == 1).group_by("player").agg(pl.col("behavior_family").n_unique())
print("positive players involved in >1 family:", (fam_per_player["behavior_family"] > 1).sum())

pos_players = set(long.filter(pl.col("label") == 1)["player"])
all_labeled = set(long["player"])
print("eval pairs containing a labeled-POSITIVE player:", eval_pairs.filter(
    pl.col("player_1").is_in(pos_players) | pl.col("player_2").is_in(pos_players)).height)
print("eval pairs containing a labeled-NEGATIVE-only player:", eval_pairs.filter(
    pl.col("player_1").is_in(all_labeled - pos_players) | pl.col("player_2").is_in(all_labeled - pos_players)).height)

📊 **What we saw:**
- 1,860 labeled pairs: **372 positives** (148 directed transfer, 132 soft play, 92 coordinated isolation) and **1,488 confirmed non-targets**. That's out of **174,000** possible same-table pairs, so about 1% is labeled.
- Labels never cross tables. 397 of 400 tables have a labeled pair. Some tables hold up to 5 colluding pairs.
- **Players repeat across pairs.** 49 positive players are in 2–3 colluding pairs, a hub or ring, and **30 positive players collude in more than one way** (different families with different partners).
- No evaluation pair contains a labeled-positive player, as the rules promise. But 42,539 evaluation pairs contain a player we have seen labeled as a non-target.

💡 **Inference 1, unknown ≠ negative.** The ~172,000 unlabeled pairs are **unknown**. Some are surely colluders the organizer didn't disclose (the description says so). If we trained with "unlabeled = innocent", we'd teach the model that hidden colluders are innocent. **So we train only on the 1,860 labeled pairs.** Using the unlabeled ones carefully (PU learning) is a learning-period experiment (L4).

💡 **Inference 2, how to validate.** Validation must imitate the real test: **evaluation pairs contain players the model has never seen as colluders.** Suppose we split pairs randomly. A hub player colluding with partners X and Y could have (hub, X) in training and (hub, Y) in validation. The model would recognize the *hub's* habits and look brilliant, but that skill doesn't exist on Kaggle, because positive players are removed from evaluation. This is **leakage**. Splitting by **table** puts every pair of a table, and so every appearance of a player, on one side of the split.

💡 **Inference 3, families aren't exclusive per player.** One player can soft-play with one partner and dump chips to another. Features and family predictions must be about the **pair**, not the individual.

➡️ **Consequence:** DEC-010 (5-fold cross-validation grouped by `table_id`); training on labeled pairs only; pair-level (not player-level) behavior modeling.

## 5 · Exposure: does playing together a lot mean anything?

🎯 **Question:** Do colluding pairs simply share more hands? And do pairs share as many hands in the evaluation period as in development?

🤔 **Why it matters:** If exposure alone predicted collusion, a lazy model would learn "flag pairs who play together a lot", which is a classic spurious shortcut. And if exposure differs between periods, any *count*-based feature would shift between training and test.

🔬 **Method:** Count shared hands per labeled pair, by period. Then compute the AP of ranking pairs *only* by shared-hand count. This is baseline B0.

In [ ]:
labeled_players = list(all_labeled)
s_dev = (seats_lf.filter(pl.col("player_id").is_in(labeled_players))
         .select("hand_id", "player_id").collect()
         .join(hands.select("hand_id", "phase", "started_at"), on="hand_id"))
pair_keys = labels.select("pair_id", "player_1", "player_2", "label", "behavior_family")
shared = (s_dev.rename({"player_id": "player_1"})
          .join(s_dev.select("hand_id", pl.col("player_id").alias("player_2")), on="hand_id")
          .join(pair_keys, on=["player_1", "player_2"]))
exp = shared.group_by("pair_id", "phase").len().pivot(on="phase", index="pair_id", values="len").fill_null(0)
exp = exp.join(pair_keys, on="pair_id")
print(exp.group_by("label").agg(pl.col("development").median().alias("dev_median"),
                                 pl.col("evaluation").median().alias("eval_median")))
print("evaluation_pairs.csv shared_hands: median", eval_pairs["shared_hands"].median(), "| min", eval_pairs["shared_hands"].min())
yl = exp["label"].to_numpy()
print(f"base rate (share of positives): {yl.mean():.3f}")
print(f"AP of ranking by shared-hand count: {average_precision(yl, exp['development'].to_numpy().astype(float)):.3f}")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(exp["development"], bins=60, alpha=.6, label="labeled pairs, development period", density=True)
ax.hist(eval_pairs["shared_hands"], bins=60, alpha=.6, label="evaluation pairs, evaluation period", density=True)
ax.legend(); ax.set_title("shared hands per pair"); plt.show()

📊 **What we saw:**
- Positives and negatives share almost the same number of development hands (median ~112 vs ~108).
- Ranking by shared hands gives **AP 0.204 at a base rate of 0.20**. A random ranking gets AP ≈ base rate, so this is **no better than random**.
- Evaluation-period exposure is about **40% lower** (labeled pairs: median ~65 vs ~110). The evaluation period is only 40% of each table's timeline.

💡 **Inference 1, the generator controls for exposure.** The organizer clearly balanced exposure on purpose (the description mentions "repeated opponent selection" as a decoy). Any model leaning on "how much they play together" will fail. The signal has to come from **how** pairs play against each other.

💡 **Inference 2, counts shift and rates don't.** "Folded to partner 6 times" means different things after 110 vs 65 shared hands. But "folded to partner in 30% of confrontations" is comparable. **So features must be rates.**

💡 **Inference 3, small samples need shrinkage.** With few hands a rate is noisy. "1 fold out of 2 confrontations" (50%) is much weaker evidence than "20 out of 40" (also 50%). We *shrink* each rate toward the player's normal behavior, by an amount that shrinks as data grows:
`shrunk = (events + k × player_baseline) / (opportunities + k)`
With k = 20 and a baseline of 25%: 1/2 becomes 27% (barely moved from normal), while 20/40 becomes 42% (clearly abnormal).

➡️ **Consequence:** DEC-011 (rates with Beta shrinkage toward the player's own baseline; the "lift" principle in SPEC §5.3). B0 is our floor baseline.

## 6 · Evidence hands: what makes a hand suspicious?

🎯 **Question:** How do the organizer's planted **evidence hands** differ from the same pair's ordinary shared hands?

🤔 **Why it matters:** Evidence ranking is 20% of the score, and these hands show what each behavior *physically looks like*. Comparing against the **same pair's** other hands controls for who the players are. Any difference is about the *hand*, not the people.

🔬 **Method:** For every positive pair, label each shared development hand as evidence or not, then compare simple hand facts: did both put money in, did one win while the other lost, pot size, how far the hand went.

In [ ]:
pos_shared = shared.filter((pl.col("label") == 1) & (pl.col("phase") == "development"))
ev_set = evidence.select("pair_id", "hand_id", pl.lit(True).alias("is_evidence"))
pos_shared = pos_shared.join(ev_set, on=["pair_id", "hand_id"], how="left").with_columns(pl.col("is_evidence").fill_null(False))
print("evidence hands found among shared dev hands:", pos_shared["is_evidence"].sum(), "of", evidence.height)

needed = pos_shared.select("hand_id").unique()
s_need = seats_lf.join(needed.lazy(), on="hand_id", how="semi").collect()
def side(i):
    return s_need.select("hand_id", pl.col("player_id").alias(f"player_{i}"), pl.col("net_chips").alias(f"net_{i}"),
                         pl.col("folded").alias(f"fold_{i}"), pl.col("went_to_showdown").alias(f"sd_{i}"),
                         pl.col("total_contribution").alias(f"in_{i}"))
hctx = (pos_shared.join(side(1), on=["hand_id", "player_1"]).join(side(2), on=["hand_id", "player_2"])
    .join(hands.select("hand_id", "big_blind", "final_pot", "board_cards"), on="hand_id")
    .with_columns(
        (pl.col("final_pot") / pl.col("big_blind")).alias("pot_bb"),
        pl.when(pl.col("board_cards") == "").then(0).otherwise(pl.col("board_cards").str.split(" ").list.len()).alias("board_n"),
        ((pl.col("in_1") > 0) & (pl.col("in_2") > 0)).alias("both_put_in"),
        (pl.col("sd_1") & pl.col("sd_2")).alias("both_showdown"),
        (((pl.col("net_1") > 0) & (pl.col("net_2") < 0)) | ((pl.col("net_2") > 0) & (pl.col("net_1") < 0))).alias("one_wins_other_loses"),
    ))
(hctx.group_by("behavior_family", "is_evidence")
    .agg(pl.len().alias("hands"),
         pl.col("both_put_in").mean().round(2), pl.col("one_wins_other_loses").mean().round(2),
         pl.col("both_showdown").mean().round(2), pl.col("pot_bb").median().round(1).alias("median_pot_bb"),
         pl.col("board_n").mean().round(1).alias("avg_board_cards"))
    .sort("behavior_family", "is_evidence"))

📊 **How to read the table:** each family has two rows, `false` (ordinary shared hands) and `true` (evidence hands). Compare them *within a family*.

| | Directed transfer | Soft play | Coordinated isolation |
|---|---|---|---|
| both partners put chips in | **100%** vs 34% | **99%** vs 39% | **71%** vs 36% |
| one wins, the other loses | **100%** vs 19% | 84% vs 19% | 44% vs 21% |
| median pot | **34.5 bb** vs 10.5 | 17 bb vs 10 | 18 bb vs 10 |
| both reach showdown | 25% vs 6% | **35%** vs 4% | 3% vs 2% |

💡 **Inference, per family:**
- **Directed transfer** evidence is *always* a hand where both are involved and money moves from one to the other, in big pots that go deep (more board cards). That's exactly what a deliberate transfer needs: the loser must put chips in and the partner must collect. **Measuring chip flow between partners, in bb, is a direct feature.**
- **Soft play** also involves both partners, but they reach **showdown together** far more often (35%). That fits *checking it down* instead of betting into each other. "Both in, but nobody attacks" is the signature.
- **Coordinated isolation** evidence often has only *one* partner putting chips in (71% both). The partner's role can be to *pressure and step aside*. This family is less about the two partners' money and more about **third players being pushed out**.

⚠️ **Caveat, selection bias:** evidence hands were *chosen* by the organizer because the behavior is visible in them. They're the clearest examples, not typical manipulated hands. Expect real coordinated hands to be subtler on average.

➡️ **Consequence:** Hand-level features for the evidence ranker (M4): both-involved, chip flow between partners in bb, pot size, street depth, and a showdown-together flag.

## 7 · Behavior toward the partner: the core signal

🎯 **Question:** When a partner faces a bet, does it matter **who** made the bet?

🤔 **Why it matters:** This is the heart of the **lift principle**. A weak player folds to everyone, a maniac raises everyone, and a tilting player does crazy things against everyone. Those are innocent decoys. **Colluders behave differently toward one specific person.** So we compare a player's response to their *partner* with their response to *everyone else*.

🔬 **Method:**
1. For every action, find the **last aggressor**: who made the latest bet or raise that this player is now responding to. (An all-in only counts as aggression if it raises; an all-in *call* doesn't.)
2. For each action by a pair member, label the situation: *facing PARTNER*, *facing other*, or *no bet yet*.
3. Compare response rates (fold / call / raise) across three kinds of hands: **evidence hands**, **other shared hands of colluding pairs**, and hands of **confirmed innocent pairs** (the control group).

In [ ]:
sample_hands = (pos_shared.filter(~pl.col("is_evidence")).group_by("pair_id").head(40)
                .select("pair_id", "hand_id", "is_evidence"))
ev_hands = pos_shared.filter(pl.col("is_evidence")).select("pair_id", "hand_id", "is_evidence")
neg_hands = (shared.filter((pl.col("label") == 0) & (pl.col("phase") == "development"))
             .group_by("pair_id").head(15).select("pair_id", "hand_id", pl.lit(False).alias("is_evidence")))
study = pl.concat([ev_hands, sample_hands, neg_hands]).join(pair_keys, on="pair_id")
acts = (actions_lf.join(study.select("hand_id").unique().lazy(), on="hand_id", how="semi")
        .collect().sort("hand_id", "action_no"))
acts = acts.with_columns(
    pl.when(pl.col("action").is_in(["bet", "raise"]) | ((pl.col("action") == "all_in") & (pl.col("amount") > pl.col("to_call"))))
      .then(pl.col("player_id")).alias("aggr")
).with_columns(pl.col("aggr").shift(1).forward_fill().over("hand_id").alias("last_aggr"))

pa = (study.join(acts, on="hand_id")
      .filter((pl.col("player_id") == pl.col("player_1")) | (pl.col("player_id") == pl.col("player_2")))
      .with_columns(pl.when(pl.col("player_id") == pl.col("player_1")).then(pl.col("player_2"))
                      .otherwise(pl.col("player_1")).alias("partner"))
      .with_columns(pl.when(pl.col("last_aggr").is_null()).then(pl.lit("no bet yet"))
                      .when(pl.col("last_aggr") == pl.col("partner")).then(pl.lit("facing PARTNER"))
                      .otherwise(pl.lit("facing other")).alias("situation"),
                    pl.when(pl.col("label") == 0).then(pl.lit("innocent pair"))
                      .when(pl.col("is_evidence")).then(pl.lit("EVIDENCE hand"))
                      .otherwise(pl.lit("other shared hand")).alias("hand_kind")))
(pa.filter(pl.col("situation") != "no bet yet")
   .group_by("behavior_family", "hand_kind", "situation")
   .agg(pl.len().alias("n"),
        (pl.col("action") == "fold").mean().round(3).alias("fold"),
        (pl.col("action") == "call").mean().round(3).alias("call"),
        pl.col("action").is_in(["raise", "all_in"]).mean().round(3).alias("raise/all-in"))
   .sort("behavior_family", "hand_kind", "situation"))

In [ ]:
(pa.filter(pl.col("situation") == "no bet yet")
   .group_by("behavior_family", "hand_kind")
   .agg(pl.len().alias("n"), pl.col("action").is_in(["bet", "raise", "all_in"]).mean().round(3).alias("bets_when_unopened"),
        (pl.col("action") == "check").mean().round(3).alias("checks_when_unopened"))
   .sort("behavior_family", "hand_kind"))

📊 **What we saw.** First the control group: **innocent pairs** respond to each other almost like they respond to anyone (fold 56% vs 48%, raise 10% vs 9%). That's our yardstick for "normal".

| | facing PARTNER | facing other | reading |
|---|---|---|---|
| **Directed transfer**, evidence | fold 21%, **call 47%**, raise 9% | fold **0.6%**, call 7% | they *pay off* the partner and never give up against others while dumping |
| **Soft play**, evidence | fold 22%, call 37%, **raise 1.4%** | raise 10.7% | they almost **never attack** the partner |
| **Coordinated isolation**, evidence | **raise 31%** | raise 20% | re-raising the partner = the squeeze |

When nobody has bet yet: isolation partners **bet or raise 97.5%** of the time (vs 24% for innocent pairs). Soft-play partners **check 25%** (vs 7%).

💡 **Inference 1, each family has a measurable fingerprint, and it's defined *relative to the partner*.** Directed transfer means *calling* the partner where a normal player folds. Soft play means *not raising* the partner where a normal player would. Isolation means *aggression*, especially re-raising behind the partner. This validates the lift principle: compare behavior toward the partner with the same player's behavior toward others.

💡 **Inference 2, the most important finding: the signal leaks beyond the 5 evidence hands.** Look at the *other shared hands* rows, which are hands the organizer did **not** list:
- Directed-transfer partners still call the partner **40%** of the time (innocent pairs: 21%).
- Isolation partners still raise the partner **21%** (innocent pairs: 10%).
- Soft-play partners raise the partner only **3.9%** (innocent pairs: 9.8%).

There are two reasons. Coordination is **episodic**, and many unlisted hands are also manipulated, because the organizer lists at most 5. **This means pair-level rates computed over *all* shared hands carry real signal**, which is good news for the 70%-weight Pair AP.

💡 **Inference 3, why compare against innocent pairs too?** Without the control group, "facing partner: fold 21%" means nothing. Is 21% high or low? The innocent-pair rows anchor every number. That is exactly what the model gets through lift features.

⚠️ **Caveats:** these are 40 sampled non-evidence hands per positive pair and 15 per negative pair, so the numbers are estimates. And all-in calls vs raises depend on our "last aggressor" definition, which we'll unit-test in P2.

➡️ **Consequence:** the core feature families F3–F5: partner-conditioned response rates as lift over the player's baseline, aggression when unopened, and squeeze sequences. Plus the strength-aware versions in section 8.

## 8 · Walkthroughs: reading nine real evidence hands

🎯 **Goal:** turn the statistics into intuition. These are the rank-1 evidence hands of the first three pairs of each family. **P1 and P2 are the partners.** Since hole cards are visible, ask yourself at every fold: *was the folder actually ahead?*

How to read a line: `bet to 13 (+13)  [facing 4]` means this action brought the player's street total to 13 by adding 13 chips, and before acting they faced a price of 4.

In [ ]:
examples = []
for fam in FAMILIES:
    pairs = labels.filter(pl.col("behavior_family") == fam).sort("pair_id").head(3)
    for p in pairs.iter_rows(named=True):
        hid = evidence.filter((pl.col("pair_id") == p["pair_id"]) & (pl.col("evidence_rank") == 1))["hand_id"].item()
        examples.append((fam, p, hid))
ids = [e[2] for e in examples]
s_ex = seats_lf.filter(pl.col("hand_id").is_in(ids)).collect()
a_ex = actions_lf.filter(pl.col("hand_id").is_in(ids)).collect()
h_ex = hands.filter(pl.col("hand_id").is_in(ids))
for k, (fam, p, hid) in enumerate(examples, start=1):
    print("=" * 110)
    print(f"EXAMPLE {k} | {fam.upper()} | P1={p['player_1'][:7]} P2={p['player_2'][:7]}")
    print(format_hand(hid, h_ex, s_ex, a_ex, highlight={p["player_1"]: "P1", p["player_2"]: "P2"}))

### Commentary

**Directed transfer**
- **Example 1.** P2 raised preflop with Q♠T♠, and the flop T-7-5 gave P2 **top pair**. P2 checked, then **folded to P1's bet**, but P1 held K-9 with *no pair*. P2 folded the best hand to the partner. *Lesson:* hole-card visibility lets us measure "folded while ahead". Without it, this fold looks completely normal.
- **Example 2.** The same pattern. P2 (A-5) has a **pair of fives** on 2-3-7-5. P1 (K-J) has nothing, bets the turn, and **P2 folds**. Again the folder was ahead.
- **Example 3.** The *other* transfer mechanism. P1 holds A-3 (just ace-high) on 9-T-T, **bets into partner P2**, calls P2's big raise, and folds to the turn bet. P1 put about 17 bb into a pot they were badly behind in, which is "paying off". *Lesson:* transfers happen through **losing calls and bets**, not only through folds.

**Soft play**
- **Example 4.** Tiny: P2 limps T-6 and folds to P1's raise. **On its own this is perfectly innocent** (T-6 is weak). *Lesson:* a single evidence hand can look benign. The case rests on **repetition across many hands**, which is why pair-level rates matter more than any one hand.
- **Example 5.** P2 has **8-8 on an A-5-9-2 board** (a decent pair), checks the flop, and folds to P1's small turn bet while P1 holds 8-3 (nothing). P2 avoided confronting the partner with the better hand.
- **Example 6.** P1 holds Q-5 and flops a **pair of fives** on A-5-8, but folds to partner P2's bet. Then P2 battles a *third* player. The partners avoid contesting the same pot.

**Coordinated isolation**
- **Example 7.** P2 raises, and everyone including partner P1 folds. Mild on its own. The partner simply steps aside.
- **Example 8.** P1 raises with K-2 offsuit (a weak hand to open with). A third player re-raises and P1 folds. *Honest note:* the coordination here isn't obvious to us from one hand. It may be the aggression-when-unopened pattern (section 7: 97.5%).
- **Example 9.** P1 opens, P2 calls, and **everyone else folds preflop**. On the flop P1 bets and P2 raises big, and P1 folds. The pair used combined preflop pressure to isolate the pot to themselves, then resolved it between them.

💡 **Inference from the walkthroughs:**
1. **Hand strength at decision time is essential.** "Folded while ahead of partner" (examples 1, 2, 5, 6) and "put chips in while far behind" (example 3) are only visible with hole cards *and* a hand evaluator. So P2 includes a fast hand-strength step (OPEN-2).
2. **Single hands are ambiguous; patterns aren't.** Examples 4, 7 and 8 would never convince an investigator alone. This is why the benign-alternative panel exists, and why the model scores *pairs* using rates.
3. We should expect the evidence ranker to find the dramatic hands (1, 2, 3, 5) much more easily than the subtle ones (4, 7, 8).

## 9 · Summary: the reasoning chain

| # | Observation | Inference | What we did |
|---|---|---|---|
| 1 | Official metric = 70% Pair AP + 20% evidence + 10% behavior | Ranking pairs dominates. Unpredicted families collapse. | Pair AP gets top priority; always assign a family; validator checks it (DEC-007) |
| 2 | Every log invariant holds at 100%; all hole cards are visible | Every decision can be reconstructed exactly, including who was ahead | Strength-aware features; `docs/data_semantics.md` binds our game logger (DEC-009) |
| 3 | Fixed stake per table; 30 players per table; one table per player | Pairs live inside tables; money must be in bb | Per-table candidate pairs; bb units |
| 4 | Only ~1% of pairs are labeled; unlabeled ≠ innocent | Treating unlabeled pairs as negatives would teach the model wrong | Train on labeled pairs only; PU learning later (L4) |
| 5 | Hub players and multi-family players; evaluation excludes positive players | Splitting by pair would leak a player's habits across folds | Grouped CV by table (DEC-010) |
| 6 | Exposure gives AP 0.204 at base rate 0.20; evaluation exposure ~40% lower | Exposure is a decoy; counts would shift | Rates + Beta shrinkage toward the player's baseline (DEC-011) |
| 7 | Evidence hands: both involved, money moves, bigger pots | Clear hand-level markers exist | Hand-level features for the evidence ranker M4 |
| 8 | Partner-specific responses differ sharply; innocent pairs don't | Collusion = behavior *relative to the partner* | The lift principle; feature families F3–F5 |
| 9 | The signal persists in unlisted hands | Coordination is episodic, and unlisted hands are also manipulated | Pair-level rates over all shared hands; rolling windows (F6) |
| 10 | Single evidence hands can look innocent | Judgments must rest on patterns | Pair-level scoring; benign-alternative explanations |

### What we did *not* conclude (open questions for later phases)
- Whether development-period behavior predicts evaluation-period behavior *for the same pair* (relationships may switch on or off). This is untested.
- How the hidden 4th family ("other coordination") looks. By design it isn't in the labels, so we can only look for anomalies (M3).
- Whether our "last aggressor" definition is the best one. It gets unit-tested in P2.

### Self-check (try answering before reading the answers below)
1. Why is ranking pairs by shared hands useless here, even though colluders "must play together"?
2. A player folds to their partner 40% of the time. Is that suspicious? What else do you need to know?
3. Why would a random pair-level train/validation split give a misleadingly high score?
4. Why must we never treat unlabeled development pairs as innocent?
5. Example 4 (soft play) looks innocent. Why is it still listed as evidence, and what does that imply for model design?

<details><summary>Answers</summary>

1. The organizer balanced exposure between colluders and innocents (AP ≈ base rate). The signal is in *how* they play, not *how much*.
2. You need their fold rate against *everyone else* (their baseline) and how many confrontations it rests on. 40% vs a baseline of 45% is normal; 40% vs a baseline of 10% over 30 confrontations is suspicious.
3. Hub players appear in several pairs. Their habits would be learned in training and recognized in validation, but evaluation contains no previously-labeled colluders.
4. Many are undisclosed colluders. Calling them innocent trains the model to ignore real collusion and distorts validation.
5. It's part of a *pattern* across many hands. Models should score pairs from aggregated rates, and explanations should acknowledge innocent readings of single hands.
</details>